# 03 · Gold — star schema for Power BI
Builds a Kimball star schema in **`lh_gold`** from `lh_silver.dbo.trips`.

| Table | Type | Grain |
|---|---|---|
| `fact_trip` | Fact | one row per booking |
| `dim_date` | Dimension (role-playing: trip start / booking / trip end) | one row per calendar day |
| `dim_origin_location`, `dim_destination_location` | Role-playing copies of one conformed location dimension | one row per location |
| `dim_vehicle`, `dim_vehicle_type`, `dim_customer`, `dim_supplier`, `dim_driver`, `dim_material` | Dimensions | one row per member |
| `dim_trip_attributes` | Junk dimension (booking type × GPS provider × delivery status) | one row per combination |

Every dimension uses an integer surrogate key and has an **Unknown** member (key `-1`),
so the fact table never has null foreign keys (except the optional trip-end date).
Tables are written with V-Order + Optimize Write for fast Direct Lake reads, and numeric
measures use fixed decimals (not floating point) so they map cleanly to the semantic model.

In [ ]:
silver_table = "lh_silver.dbo.trips"

In [ ]:
from pyspark.sql import Window
from pyspark.sql import functions as F

spark.conf.set("spark.sql.parquet.vorder.default", "true")
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.optimizeWrite.binSize", "1g")

trips = spark.table(silver_table).cache()
print(f"Silver trips: {trips.count():,}")

UNKNOWN = "Unknown"


def write_gold(df, name):
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"dbo.{name}")
    print(f"dbo.{name}: {spark.table(f'dbo.{name}').count():,} rows")


def with_key(df, key, order_cols, unknown_row):
    """Add a stable integer surrogate key and the Unknown (-1) member."""
    keyed = df.withColumn(key, F.row_number().over(Window.orderBy(*order_cols)).cast("int"))
    unknown = spark.createDataFrame([unknown_row], schema=df.schema).withColumn(key, F.lit(-1))
    return keyed.unionByName(unknown).select(key, *df.columns)


def mode_by(df, group_col, value_cols):
    """Most frequent combination of value_cols for each group_col value."""
    counted = df.groupBy(group_col, *value_cols).count()
    w = Window.partitionBy(group_col).orderBy(F.desc("count"), *value_cols)
    return counted.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn", "count")

## Date dimension

In [ ]:
bounds = trips.select(F.least(F.min("booking_date"), F.to_date(F.min("trip_start_ts"))).alias("lo"),
                      F.greatest(F.max("booking_date"), F.to_date(F.max("trip_end_ts"))).alias("hi")).first()
dates = spark.sql(f"""
    SELECT explode(sequence(DATE'{bounds.lo.year}-01-01', DATE'{bounds.hi.year}-12-31', INTERVAL 1 DAY)) AS date
""")
dim_date = dates.select(
    F.date_format("date", "yyyyMMdd").cast("int").alias("date_key"),
    "date",
    F.year("date").alias("year"),
    F.quarter("date").alias("quarter_number"),
    F.concat(F.lit("Q"), F.quarter("date")).alias("quarter"),
    F.concat(F.year("date"), F.lit(" Q"), F.quarter("date")).alias("year_quarter"),
    (F.year("date") * 10 + F.quarter("date")).alias("year_quarter_number"),
    F.month("date").alias("month_number"),
    F.date_format("date", "MMMM").alias("month_name"),
    F.date_format("date", "MMM").alias("month_short"),
    F.date_format("date", "MMM yyyy").alias("year_month"),
    (F.year("date") * 100 + F.month("date")).alias("year_month_number"),
    F.weekofyear("date").alias("week_of_year"),
    F.dayofmonth("date").alias("day_of_month"),
    ((F.dayofweek("date") + 5) % 7 + 1).alias("day_of_week_number"),  # Monday = 1
    F.date_format("date", "EEEE").alias("day_name"),
    F.date_format("date", "EEE").alias("day_short"),
    F.dayofweek("date").isin(1, 7).alias("is_weekend"),
)
write_gold(dim_date, "dim_date")

## Location dimension (conformed, then role-played as Origin and Destination)

In [ ]:
loc_cols = ["location_code", "place", "city", "state", "latitude", "longitude"]
loc_union = (
    trips.select(F.col("origin_location").alias("location_name"), F.col("origin_location_code").alias("location_code"),
                 F.col("origin_place").alias("place"), F.col("origin_city").alias("city"),
                 F.col("origin_state").alias("state"), F.col("origin_lat").alias("latitude"),
                 F.col("origin_lon").alias("longitude"))
    .unionByName(
        trips.select(F.col("destination_location").alias("location_name"),
                     F.col("destination_location_code").alias("location_code"),
                     F.col("destination_place").alias("place"), F.col("destination_city").alias("city"),
                     F.col("destination_state").alias("state"), F.col("destination_lat").alias("latitude"),
                     F.col("destination_lon").alias("longitude")))
    .filter(F.col("location_name").isNotNull())
)
locations = (mode_by(loc_union, "location_name", loc_cols)
             .withColumn("latitude", F.col("latitude").cast("decimal(9,4)"))
             .withColumn("longitude", F.col("longitude").cast("decimal(9,4)"))
             .withColumn("city", F.coalesce("city", F.lit(UNKNOWN)))
             .withColumn("state", F.coalesce("state", F.lit(UNKNOWN))))
dim_location = with_key(locations, "location_key", ["location_name"],
                        (UNKNOWN, None, UNKNOWN, UNKNOWN, UNKNOWN, None, None))
write_gold(dim_location, "dim_origin_location")
write_gold(dim_location, "dim_destination_location")

## Vehicle, vehicle type, customer, supplier, driver, material

In [ ]:
RTO_STATES = {
    "AP": "Andhra Pradesh", "AR": "Arunachal Pradesh", "AS": "Assam", "BR": "Bihar", "CG": "Chhattisgarh",
    "CH": "Chandigarh", "DD": "Daman & Diu", "DL": "Delhi", "DN": "Dadra & Nagar Haveli", "GA": "Goa",
    "GJ": "Gujarat", "HP": "Himachal Pradesh", "HR": "Haryana", "JH": "Jharkhand", "JK": "Jammu & Kashmir",
    "KA": "Karnataka", "KL": "Kerala", "MH": "Maharashtra", "ML": "Meghalaya", "MN": "Manipur",
    "MP": "Madhya Pradesh", "MZ": "Mizoram", "NL": "Nagaland", "OD": "Odisha", "OR": "Odisha",
    "PB": "Punjab", "PY": "Puducherry", "RJ": "Rajasthan", "SK": "Sikkim", "TN": "Tamil Nadu",
    "TR": "Tripura", "TS": "Telangana", "UK": "Uttarakhand", "UP": "Uttar Pradesh", "WB": "West Bengal",
}
rto_map = F.create_map(*[F.lit(x) for kv in RTO_STATES.items() for x in kv])
vehicles = (trips.filter(F.col("vehicle_no").isNotNull()).select("vehicle_no").distinct()
            .withColumn("registration_state_code", F.substring("vehicle_no", 1, 2))
            .withColumn("registration_state", F.coalesce(rto_map[F.col("registration_state_code")], F.lit(UNKNOWN))))
dim_vehicle = with_key(vehicles, "vehicle_key", ["vehicle_no"], (UNKNOWN, None, UNKNOWN))
write_gold(dim_vehicle, "dim_vehicle")

vt = F.upper(F.col("vehicle_type"))
capacity = F.regexp_extract("vehicle_type", r"(\d+(?:\.\d+)?)\s*MT", 1)
length = F.regexp_extract("vehicle_type", r"(\d+)\s*FT", 1)
vehicle_types = (
    trips.filter(F.col("vehicle_type").isNotNull()).select("vehicle_type").distinct()
    .withColumn("capacity_mt", F.when(capacity != "", capacity.cast("decimal(6,2)")))
    .withColumn("length_ft", F.when(length != "", length.cast("int")))
    .withColumn("vehicle_class",
                F.when(vt.contains("TRAILER") | vt.contains("FLAT BED"), "Trailer")
                 .when(vt.contains("LCV") | vt.contains("ACE") | vt.contains("PICKUP"), "LCV")
                 .when(vt.contains("MCV"), "MCV")
                 .when(vt.contains("HCV"), "HCV")
                 .when(F.col("capacity_mt") >= 12, "HCV")
                 .when(F.col("capacity_mt") >= 4, "MCV")
                 .when(F.col("capacity_mt").isNotNull(), "LCV")
                 .otherwise("Other"))
    .withColumn("body_style",
                F.when(vt.contains("FLAT BED"), "Flat Bed")
                 .when(vt.contains("CONTAINER") | vt.contains("CLOSE"), "Closed / Container")
                 .when(vt.contains("TRAILER"), "Trailer")
                 .when(vt.contains("OPEN") | vt.contains("TAURUS") | vt.contains("TARUS"), "Open")
                 .otherwise("Other"))
)
dim_vehicle_type = with_key(vehicle_types, "vehicle_type_key", ["vehicle_type"], (UNKNOWN, None, None, UNKNOWN, UNKNOWN))
write_gold(dim_vehicle_type, "dim_vehicle_type")

customers = mode_by(trips.filter(F.col("customer_id").isNotNull()), "customer_id", ["customer_name"]) \
    .withColumn("customer_name", F.initcap("customer_name"))
dim_customer = with_key(customers, "customer_key", ["customer_id"], (UNKNOWN, UNKNOWN))
write_gold(dim_customer, "dim_customer")

suppliers = mode_by(trips.filter(F.col("supplier_id").isNotNull()), "supplier_id", ["supplier_name"]) \
    .withColumn("supplier_name", F.initcap("supplier_name"))
dim_supplier = with_key(suppliers, "supplier_key", ["supplier_id"], (UNKNOWN, UNKNOWN))
write_gold(dim_supplier, "dim_supplier")

drivers = trips.filter(F.col("driver_name").isNotNull()).select("driver_name").distinct()
dim_driver = with_key(drivers, "driver_key", ["driver_name"], (UNKNOWN,))
write_gold(dim_driver, "dim_driver")

materials = trips.filter(F.col("material_shipped").isNotNull()).select("material_shipped").distinct()
dim_material = with_key(materials, "material_key", ["material_shipped"], (UNKNOWN,))
write_gold(dim_material, "dim_material")

## Junk dimension: trip attributes

In [ ]:
attr_cols = ["booking_type", "gps_provider", "delivery_status"]
trip_attr_src = (trips
                 .withColumn("booking_type", F.coalesce("booking_type", F.lit(UNKNOWN)))
                 .withColumn("gps_provider", F.coalesce("gps_provider", F.lit("NOT TRACKED"))))
attributes = (trip_attr_src.select(*attr_cols).distinct()
              .withColumn("is_gps_tracked", F.col("gps_provider") != "NOT TRACKED"))
dim_trip_attributes = with_key(attributes, "trip_attributes_key", attr_cols, (UNKNOWN, UNKNOWN, UNKNOWN, False))
write_gold(dim_trip_attributes, "dim_trip_attributes")

## Fact table

In [ ]:
def lookup(df, dim_table, natural_cols, key, fact_cols=None):
    """Left-join a dimension on its natural key(s); missing members map to -1."""
    fact_cols = fact_cols or natural_cols
    dim = spark.table(f"dbo.{dim_table}").filter(F.col(key) != -1).select(
        key, *[F.col(n).alias(f) for n, f in zip(natural_cols, fact_cols)])
    return df.join(F.broadcast(dim), on=fact_cols, how="left").withColumn(key, F.coalesce(key, F.lit(-1)))


date_key = lambda c: F.date_format(c, "yyyyMMdd").cast("int")
fact = trip_attr_src
fact = lookup(fact, "dim_origin_location", ["location_name"], "location_key", ["origin_location"]) \
    .withColumnRenamed("location_key", "origin_location_key")
fact = lookup(fact, "dim_destination_location", ["location_name"], "location_key", ["destination_location"]) \
    .withColumnRenamed("location_key", "destination_location_key")
fact = lookup(fact, "dim_vehicle", ["vehicle_no"], "vehicle_key")
fact = lookup(fact, "dim_vehicle_type", ["vehicle_type"], "vehicle_type_key")
fact = lookup(fact, "dim_customer", ["customer_id"], "customer_key")
fact = lookup(fact, "dim_supplier", ["supplier_id"], "supplier_key")
fact = lookup(fact, "dim_driver", ["driver_name"], "driver_key")
fact = lookup(fact, "dim_material", ["material_shipped"], "material_key")
fact = lookup(fact, "dim_trip_attributes", attr_cols, "trip_attributes_key")

fact_trip = fact.select(
    "booking_id",
    date_key("trip_start_ts").alias("trip_start_date_key"),
    date_key("booking_date").alias("booking_date_key"),
    date_key("trip_end_ts").alias("trip_end_date_key"),
    "origin_location_key", "destination_location_key", "vehicle_key", "vehicle_type_key",
    "customer_key", "supplier_key", "driver_key", "material_key", "trip_attributes_key",
    "trip_start_ts", "trip_end_ts", "actual_eta_ts",
    *[F.col(c).cast("decimal(10,2)").alias(c)
      for c in ("distance_km", "min_km_per_day", "trip_duration_hours", "transit_hours")],
    (F.col("delivery_status") == "On Time").cast("int").alias("is_on_time"),
    (F.col("delivery_status") == "Delayed").cast("int").alias("is_delayed"),
    F.col("is_completed").cast("int").alias("is_completed"),
)
write_gold(fact_trip, "fact_trip")
spark.sql("OPTIMIZE dbo.fact_trip")

## Validate the star schema

In [ ]:
f = spark.table("dbo.fact_trip")
n_fact, n_silver = f.count(), trips.count()
assert n_fact == n_silver, f"fact_trip has {n_fact} rows, Silver has {n_silver}"
assert f.select("booking_id").distinct().count() == n_fact, "booking_id must be unique in fact_trip"

checks = {
    "trip_start_date_key": ("dim_date", "date_key"), "booking_date_key": ("dim_date", "date_key"),
    "trip_end_date_key": ("dim_date", "date_key"),
    "origin_location_key": ("dim_origin_location", "location_key"),
    "destination_location_key": ("dim_destination_location", "location_key"),
    "vehicle_key": ("dim_vehicle", "vehicle_key"), "vehicle_type_key": ("dim_vehicle_type", "vehicle_type_key"),
    "customer_key": ("dim_customer", "customer_key"), "supplier_key": ("dim_supplier", "supplier_key"),
    "driver_key": ("dim_driver", "driver_key"), "material_key": ("dim_material", "material_key"),
    "trip_attributes_key": ("dim_trip_attributes", "trip_attributes_key"),
}
rows = []
for fk, (dim, pk) in checks.items():
    keys = spark.table(f"dbo.{dim}").select(F.col(pk).alias(fk))
    orphans = f.filter(F.col(fk).isNotNull()).join(keys, fk, "left_anti").count()
    unknown = f.filter(F.col(fk) == -1).count()
    nulls = f.filter(F.col(fk).isNull()).count()
    rows.append((fk, dim, orphans, unknown, nulls))
    assert orphans == 0, f"{fk} has {orphans} orphan rows"
display(spark.createDataFrame(rows, "foreign_key string, dimension string, orphans long, unknown_members long, nulls long"))
print(f"fact_trip: {n_fact:,} rows - all foreign keys resolve")